# SafeMed AI: train the Stage 2 filter (3 seeds) on a Colab GPU

1. **Runtime > Change runtime type > T4 GPU**, then Save.
2. **Runtime > Run all**. When cell 1 asks, upload `filter_colab_bundle.zip`.
3. The last cell downloads `safemed_filter_v2.zip`. Send it back (put it in `backend/models/`).

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()
name = next(iter(uploaded))
zipfile.ZipFile(name).extractall('/content/safemed')
os.chdir('/content/safemed')
print(os.listdir('.'))

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime > Change runtime type > T4 GPU')
!pip -q install -U transformers scikit-learn

In [ ]:
!python -m src.stage2_filter.train_filter \
  --train data/annotations/pqaa_filter_train.jsonl \
  --val data/annotations/pqaa_filter_val.jsonl \
  --output-dir models/safemed_filter_v2 \
  --seeds 42 43 44

In [ ]:
import json
s = json.load(open('models/safemed_filter_v2/seeds_summary.json'))
print(json.dumps(s, indent=1)[:3000])

In [ ]:
import shutil, glob
out = 'models/safemed_filter_v2'
with zipfile.ZipFile('/content/safemed_filter_v2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in glob.glob(out + '/**/*', recursive=True):
        if os.path.isfile(path) and not (path.endswith('.safetensors') and '/seed_' in path):
            z.write(path, os.path.relpath(path, 'models'))
print(round(os.path.getsize('/content/safemed_filter_v2.zip') / 1e6, 1), 'MB')
files.download('/content/safemed_filter_v2.zip')